# PopGenLM Bench · verified analysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahirali-biomics/tahirali-biomics.github.io/blob/main/notebooks/popgenlm-bench-demo.ipynb)

This notebook independently checks and summarises the public 100-variant engineering fixture used on the PopGenLM Bench project page. It uses only public data and standard packages available in Google Colab.

**Interpretation boundary:** the fixture verifies data flow, score handling, statistics, and reporting. It is not a sampled population and is not evidence for selection or functional constraint.


## 1. Load the verified public fixture

The notebook first tries the public raw file. A compressed copy is embedded as a deterministic fallback, so the analysis still runs if the network is temporarily unavailable.


In [ ]:
from __future__ import annotations

import base64
import gzip
import io
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/tahirali-biomics/tahirali-biomics.github.io/main/assets/popgenlm/gpn-scores-100.tsv"
EMBEDDED_FIXTURE_GZ_B64 = "H4sICFkrl2oAA2dwbi1zY29yZXMtMTAwLnRzdgCtWs1qnEkMPM8+RV4gjf7V2lvIIS+QuzGTCTY4mWVsw+7br779uWoOKWzwwTCFWtXVVdKcn27XH6c/rq+n2+X76fHl7fT9+c+399vl9Hp9v50vDy/X8/vr6fV8vV1+Oz/d/OHb5cf1QT2iiI6/GUQn2XT6evp0+nZ5u9x+PP98fn17Pj/0xz2c/zq/XB4+ff7S/14f3p4eX54ffz5+OD7q9/8+5eP/n0KLQral8wjFAKiPB5ZJSsxY0kiffxFLFlMqh49ICkCiFRTBESOSYVqV6pyiI5RjWqU7w5PmAwwMVpdERJwjViKweGmVes1l7WbFFwAUVzdsJkaBLpaYmdZ4goXQi8YqtnIayyrGnKA7O9fI90LIBS2WLPc98qK0a/r66wdoycI0imBhBIN3KbPOB+iAXh0HuEPFaoQKQK+aFlys6XOvEtErXmW+N8kItRFVHbTwzNgjVGGqimRjn3rVgoypau+wzKkqJZC7EOnji5yapSSYZ8T2VtqT3iopBqplqVJshDKM3lq1Cu75AB1DQed+sHKuCiIX2r0y89EKKiUCKhbbbls5IkHUwhenB+VMwEKwwtvgshGPVTEhoGIJl3DOUBBvYcvaS7PPUBCx0NXk2zx6TmVEFjneq5Q2leMNZsPc4F0VI/8YEUU6iXDf3jFfddBDcEJW7dpxhxMgYxEWza9R/xhnLHjnaKM7qoCCyG7LKTJaC4EEkSaguPrcLYFYC2l3e7yMI5KAIk+11Op4gQWUQ2Kn+ZhP29BjKKjVudtmvyQQZ9GOkzc12AiFcBZtOGv3vRotuwpkbtFVlXj/zhyEDC50ta+gzqgjFMJbtDRJFMccDxSURDT6FssUu1UZwgsijna385CkLzmiW8dQVc15ttKKyCItTklOd6pCKAav4w3x+RKrQ5qVVU4yTn5UIf6Cl1D/zPlAQXNOkTjC8AgFkQtZwZvnZ0QR7uJwgiFEo1gYAR6Rpl8y7dHEGGjEWZs4a64JNLMISdGZEqYgTdptlHXvkRWGmFo0lldL+7jAaoHEvMLcUPvOI2KIrQitRqrux6hLhhhbyKLaxzRmRNqgYZYRec5FQcYWtqR94JwOHBREKt1iXF+pg+Si+3QME0YoxEakoci49jyLcUgSsTa3cUcrHDK0iMXd9G0zFEQrsl10q62Pyu6oCWenqzsWxiETzpYl6WQgM9chc4smIFfmng+wMFW59QMyK2CAvm6RnT5l3r0EZGrRuuQcs1sPSAaR1VJBOmtFgNYhYXlvcBaQdcg/76K4jTk4EEMLXru1wmSkeoAySHTmZp75B8ogpbtsXrwEJIN0ijsWf3dGt4HYntKyZkVyjvcqQTML6bzT5nWkYDKkLKnDW9B4gomyFnyo4EiMhCxEWtm3C7vNWKD1qZO1so+3OCHm4t8NPs0+OlFDi93uNWZeoMyFpezx64GaIHPRecdsdmcJ2Ykcs6w6hmZ/A1YgmFT4KQAA"

try:
    scores = pd.read_csv(DATA_URL, sep="\t")
    data_source = DATA_URL
except Exception:
    raw = gzip.decompress(base64.b64decode(EMBEDDED_FIXTURE_GZ_B64)).decode("utf-8")
    scores = pd.read_csv(io.StringIO(raw), sep="\t")
    data_source = "embedded verified fallback"

print(f"Loaded {len(scores)} rows from {data_source}")
scores.head()


## 2. Validate schema and alleles

These checks fail loudly if required columns are missing, positions are invalid, alleles are not single canonical bases, scores are non-numeric, or variant rows are duplicated.


In [ ]:
REQUIRED = {"chrom", "pos", "ref", "alt", "score"}
missing = REQUIRED.difference(scores.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

scores = scores.copy()
scores["pos"] = pd.to_numeric(scores["pos"], errors="raise").astype(int)
scores["score"] = pd.to_numeric(scores["score"], errors="raise").astype(float)
assert (scores["pos"] > 0).all(), "All positions must be positive integers"
assert scores["ref"].str.fullmatch("[ACGT]").all(), "REF must contain single canonical bases"
assert scores["alt"].str.fullmatch("[ACGT]").all(), "ALT must contain single canonical bases"
assert (scores["ref"] != scores["alt"]).all(), "REF and ALT must differ"
assert np.isfinite(scores["score"]).all(), "Scores must be finite"
assert not scores.duplicated(["chrom", "pos", "ref", "alt"]).any(), "Duplicate variants detected"

print("✓ Schema, coordinates, alleles, scores, and uniqueness checks passed")


## 3. Recalculate the statistical summary

The bootstrap interval describes uncertainty in the fixture median. It should not be confused with the biological uncertainty of a population estimate.


In [ ]:
def summarise_scores(frame: pd.DataFrame, n_boot: int = 2_000, seed: int = 42) -> dict:
    values = frame["score"].to_numpy(float)
    rng = np.random.default_rng(seed)
    boot = np.median(rng.choice(values, size=(n_boot, len(values)), replace=True), axis=1)
    q1, median, q3 = np.quantile(values, [0.25, 0.50, 0.75])
    ci_low, ci_high = np.quantile(boot, [0.025, 0.975])
    return {
        "n_variants": int(len(values)),
        "minimum": float(values.min()),
        "q1": float(q1),
        "median": float(median),
        "mean": float(values.mean()),
        "q3": float(q3),
        "maximum": float(values.max()),
        "standard_deviation": float(values.std(ddof=1)),
        "proportion_negative": float((values < 0).mean()),
        "median_bootstrap_95pct_ci": [float(ci_low), float(ci_high)],
        "bootstrap_replicates": int(n_boot),
        "bootstrap_seed": int(seed),
        "interpretation_boundary": "Engineering fixture; not a population sample or evidence for selection or constraint.",
    }

summary = summarise_scores(scores)
pd.Series(summary)


## 4. Create a publication-quality benchmark overview

Every panel uses the same validated score table: distribution, position within the fixture, ranked profile, and interval summary.


In [ ]:
NAVY, BLUE, TEAL, GOLD = "#0D2238", "#1972BD", "#14827A", "#D79B2E"
SLATE, GRID, PALE_TEAL = "#5D6F80", "#DDE6ED", "#E8F6F3"

def clean_axis(ax, label, title):
    ax.set_facecolor("white")
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#BFCBD6")
    ax.tick_params(colors=SLATE, labelsize=9)
    ax.text(0, 1.125, label, transform=ax.transAxes, color=TEAL, fontsize=9, fontweight="bold")
    ax.set_title(title, loc="left", color=NAVY, fontsize=12, fontweight="bold", pad=4, y=1.015)

values = scores["score"].to_numpy(float)
positions = scores["pos"].to_numpy(int)
median = summary["median"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9), dpi=150, facecolor="#F7F9FC")
fig.subplots_adjust(left=0.07, right=0.985, top=0.82, bottom=0.12, hspace=0.67, wspace=0.30)
fig.suptitle("PopGenLM Bench · verified engineering fixture", x=0.02, y=0.97, ha="left", fontsize=20, fontweight="bold", color=NAVY)
fig.text(0.02, 0.92, "100 deterministic SNVs · GPN Brassicales · alternate-minus-reference log-likelihood ratio", color=SLATE, fontsize=10.5)

ax = axes[0, 0]
clean_axis(ax, "A · DISTRIBUTION", "Most fixture scores fall below zero")
bins = np.linspace(values.min() - 0.15, values.max() + 0.15, 18)
counts, edges, patches = ax.hist(values, bins=bins, edgecolor="white", linewidth=1)
for patch, left, right in zip(patches, edges[:-1], edges[1:]):
    patch.set_facecolor(TEAL if (left + right) / 2 < 0 else GOLD)
ax.axvline(0, color=SLATE, linestyle=(0, (4, 3)))
ax.axvline(median, color=NAVY, linewidth=2)
ax.set(xlabel="GPN score", ylabel="Variant count")
ax.text(0.97, 0.92, f"median  {median:.3f}\nnegative  {summary['proportion_negative']:.0%}", transform=ax.transAxes, ha="right", va="top", bbox={"boxstyle":"round,pad=0.5", "facecolor":"white", "edgecolor":GRID})

ax = axes[0, 1]
clean_axis(ax, "B · POSITION", "Scores across the deterministic test locus")
colors = np.where(values < 0, TEAL, GOLD)
ax.plot(positions, values, color="#C7D3DD", linewidth=0.8)
ax.scatter(positions, values, c=colors, s=24, edgecolor="white", linewidth=0.5, zorder=2)
ax.axhline(0, color=SLATE, linestyle=(0, (4, 3)))
ax.set(xlabel="Position within fixture", ylabel="GPN score")
ax.margins(x=0.03, y=0.12)

ax = axes[1, 0]
clean_axis(ax, "C · RANK", "Lower tail and positive-score set")
ranked = np.sort(values); ranks = np.arange(1, len(ranked) + 1); neg = ranked < 0
ax.fill_between(ranks, ranked, 0, where=neg, color=PALE_TEAL)
ax.plot(ranks[neg], ranked[neg], color=TEAL, linewidth=2.2)
ax.plot(ranks[~neg], ranked[~neg], color=GOLD, linewidth=2.2)
ax.axhline(0, color=SLATE, linestyle=(0, (4, 3)))
ax.set(xlabel="Variant rank", ylabel="GPN score", xlim=(1, len(ranked)))

ax = axes[1, 1]
clean_axis(ax, "D · INTERVALS", "Uncertainty is separate from score spread")
ax.grid(False)
ci_low, ci_high = summary["median_bootstrap_95pct_ci"]
ax.hlines(3, summary["minimum"], summary["maximum"], color="#AEBCC8", linewidth=7)
ax.hlines(2, summary["q1"], summary["q3"], color=BLUE, linewidth=9)
ax.hlines(1, ci_low, ci_high, color=TEAL, linewidth=9)
ax.scatter([median, median], [2, 1], color=NAVY, marker="D", s=55, zorder=3)
ax.axvline(0, color=SLATE, linestyle=(0, (4, 3)))
ax.set_yticks([3, 2, 1], ["Observed range", "Interquartile range", "Median 95% bootstrap CI"])
ax.set_xlabel("GPN score")

fig.text(0.02, 0.025, "Engineering fixture only · not a population sample and not evidence for selection or functional constraint", color=SLATE, fontsize=9)
FIGURE_PATH = Path("popgenlm_benchmark_colab.png")
fig.savefig(FIGURE_PATH, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


## 5. Optional: analyse your own compatible score table

Set `ANALYSE_UPLOAD = True` and run the cell. The uploaded TSV or CSV must contain `chrom`, `pos`, `ref`, `alt`, and `score`. This performs the same schema checks and summary calculation; it does **not** run model inference.


In [ ]:
ANALYSE_UPLOAD = False

if ANALYSE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one TSV or CSV file")
    name, content = next(iter(uploaded.items()))
    separator = "\t" if name.lower().endswith((".tsv", ".txt")) else ","
    user_scores = pd.read_csv(io.BytesIO(content), sep=separator)
    missing = REQUIRED.difference(user_scores.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    user_scores["pos"] = pd.to_numeric(user_scores["pos"], errors="raise").astype(int)
    user_scores["score"] = pd.to_numeric(user_scores["score"], errors="raise").astype(float)
    if not user_scores["ref"].str.fullmatch("[ACGT]").all() or not user_scores["alt"].str.fullmatch("[ACGT]").all():
        raise ValueError("REF and ALT must contain single canonical bases")
    user_summary = summarise_scores(user_scores)
    display(pd.Series(user_summary))
else:
    print("Using the verified fixture. Set ANALYSE_UPLOAD = True to upload a compatible score table.")


## 6. Export checked outputs

The validated table, recalculated summary, and benchmark figure are written to the Colab session. Uncomment the final two lines to download a ZIP archive.


In [ ]:
OUTPUT_DIR = Path("popgenlm_verified_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
scores.to_csv(OUTPUT_DIR / "validated_scores.tsv", sep="\t", index=False)
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
(OUTPUT_DIR / FIGURE_PATH.name).write_bytes(FIGURE_PATH.read_bytes())

archive = Path("popgenlm_verified_outputs.zip")
import shutil
shutil.make_archive(archive.with_suffix("").as_posix(), "zip", OUTPUT_DIR)
print(f"Created {archive} with validated table, summary, and figure")

# from google.colab import files
# files.download(str(archive))


## What this notebook does not do

It does not download model weights or score a new genome. Full GPN inference requires a reference sequence, a variant table, pinned model and code revisions, and substantially more compute. That workflow belongs in the versioned PopGenLM Bench package; this notebook provides a stable, inspectable analysis layer that can run on CPU in a clean Colab session.
